# Modèle de boosting

La base d'apprentissage est construite à partir de `data/sample.csv`.
Une observation est un couple session–produit. La cible `purchased` vaut 1 si ce produit est acheté dans la session.
Les variables décrivent le produit (vues, catégorie, sous-catégorie, marque, prix) et la session (durée en secondes, historique, heure).
Fichiers écrits : `data/dataset.csv`, `data/primary1.csv`, `data/X.csv`, `data/y.csv`, `data/X_train.csv`, `data/X_test.csv`, `data/y_train.csv`, `data/y_test.csv`.
Un LightGBM est comparé en validation croisée, puis ajusté sur un partage 75 % / 25 %.


## Chargement


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set()

data = pd.read_csv(os.path.expanduser("data/sample.csv"))
print("Taille de l'échantillon :", data.shape[0])
print("Taille mémoire : {:2.2f} Mb".format(data.memory_usage().sum() / 1e6))
data.head()


## Construction de la base

Le grain est le couple session–produit.
La table d'apprentissage ne retient que les événements `cart` et `purchase`. Les vues restent dans `num_views_session` et `num_views_product`, calculés avant ce filtre.
Les doublons de type d'événement, de produit, d'utilisateur et de session sont retirés à cette étape : les comptes de vues sont déjà agrégés.


In [ ]:
# Filtre : user_session non nulle, pour éviter un groupe NaN
events_per_session = data.loc[~data["user_session"].isna(), :].copy()

events_per_session["purchased"] = np.where(events_per_session["event_type"] == "purchase", 1, 0)
# 1 si le produit est acheté au moins une fois dans la session
events_per_session["purchased"] = (
    events_per_session
    .groupby(["user_session", "product_id"])["purchased"]
    .transform("max")
)
events_per_session.sample(n=5)


In [ ]:
events_per_session["num_views_session"] = np.where(events_per_session["event_type"] == "view", 1, 0)
events_per_session["num_views_session"] = (
    events_per_session.groupby(["user_session"])["num_views_session"].transform("sum")
)

events_per_session["num_views_product"] = np.where(events_per_session["event_type"] == "view", 1, 0)
events_per_session["num_views_product"] = (
    events_per_session.groupby(["user_session", "product_id"])["num_views_product"].transform("sum")
)
events_per_session.sample(n=5)


In [ ]:
events_per_session["category"] = (
    events_per_session["category_code"].str.split(".", expand=True)[0].astype("category")
)
events_per_session["sub_category"] = (
    events_per_session["category_code"].str.split(".", expand=True)[1].astype("category")
)


In [ ]:
# Horodatage UTC
events_per_session["event_time"] = pd.to_datetime(events_per_session["event_time"], utc=True)

events_per_session["hour"] = events_per_session["event_time"].dt.hour
events_per_session["minute"] = events_per_session["event_time"].dt.minute
events_per_session["weekday"] = events_per_session["event_time"].dt.dayofweek


In [ ]:
sessions_duration = (
    events_per_session
    .groupby("user_session")
    .agg({"event_time": ["min", "max"]})
    .reset_index()
)

sessions_duration["amax"] = pd.to_datetime(sessions_duration["event_time"]["max"])
sessions_duration["amin"] = pd.to_datetime(sessions_duration["event_time"]["min"])
del sessions_duration["event_time"]
sessions_duration.columns = sessions_duration.columns.get_level_values(0)
# Unité : composante dt.seconds du timedelta
sessions_duration["duration"] = (sessions_duration["amax"] - sessions_duration["amin"]).dt.seconds
sessions_duration.head()


In [ ]:
dataset = (
    events_per_session
    .sort_values("event_time")
    .drop_duplicates(["event_type", "product_id", "user_id", "user_session"])
    .loc[events_per_session["event_type"].isin(["cart", "purchase"])]
    .merge(
        sessions_duration[["user_session", "duration"]],
        how="left",
        on="user_session",
    )
)
dataset.head()


In [ ]:
count_prev_sessions = (
    dataset[["user_id", "user_session", "event_time"]]
    .sort_values("event_time")
    .groupby(["user_id", "user_session"])
    .first()
    .reset_index()
)

# Rang de la session pour l'utilisateur, 0 pour la première
count_prev_sessions["num_prev_sessions"] = (
    count_prev_sessions
    .sort_values("event_time")
    .groupby("user_id")
    .cumcount()
)
count_prev_sessions.head()


In [ ]:
dataset = dataset.merge(
    count_prev_sessions[["user_session", "num_prev_sessions"]],
    how="left",
    on="user_session",
)


In [ ]:
view_prev_session = dataset[["user_id", "user_session", "event_time", "product_id"]]
view_prev_session = (
    view_prev_session
    .sort_values("event_time")
    .groupby(["user_id", "user_session", "product_id"])
    .first()
    .reset_index()
)

# Même rang, par couple utilisateur–produit, 0 pour la première occurrence
view_prev_session["num_prev_product_views"] = (
    view_prev_session
    .sort_values("event_time")
    .groupby(["user_id", "product_id"])
    .cumcount()
)

dataset = dataset.merge(
    view_prev_session[["user_session", "product_id", "num_prev_product_views"]],
    how="left",
    on=["user_session", "product_id"],
)


In [ ]:
dataset = (
    dataset
    .sort_values("event_time")
    .drop_duplicates(["user_session", "product_id", "purchased"])
    .drop(["event_time", "event_type", "category_code", "category_id"], axis=1)
)

dataset.to_csv(os.path.expanduser("data/dataset.csv"), index=False)
dataset.head()


In [ ]:
print("Taille du jeu de données :", dataset.shape[0])
print("Proportion des classes :")
dataset["purchased"].value_counts() / dataset.shape[0]


In [ ]:
dataset.to_csv(os.path.expanduser("data/primary1.csv"), index=False)


## Encodage

`brand`, `category` et `sub_category` sont encodées par étiquettes, puis forcées en numérique.
Les fichiers écrits ici sont ceux lus par les notebooks d'optimisation et de validation.


In [ ]:
from sklearn.preprocessing import LabelEncoder

features = dataset.drop(["user_id", "user_session"], axis=1).copy()

for label in ["category", "sub_category", "brand"]:
    features[label] = features[label].astype(str)
    # astype(str) écrit les manquants comme la chaîne "nan"
    features.loc[features[label] == "nan", label] = "unknown"
    features.loc[:, label] = LabelEncoder().fit_transform(features.loc[:, label].copy())

features["weekday"] = features["weekday"].astype(int)
features.head()


In [ ]:
features.isna().sum()


In [ ]:
from sklearn.model_selection import train_test_split

X = features.drop("purchased", axis=1)
y = features["purchased"]
columns_to_convert = ["brand", "category", "sub_category"]

for col in columns_to_convert:
    # dtype objet possible après LabelEncoder
    X[col] = pd.to_numeric(X[col], errors="coerce")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=40
)
X_train.head()


In [ ]:
X.to_csv(os.path.expanduser("data/X.csv"), index=False)
y.to_csv(os.path.expanduser("data/y.csv"), index=False)

X_train.to_csv(os.path.expanduser("data/X_train.csv"), index=False)
X_test.to_csv(os.path.expanduser("data/X_test.csv"), index=False)
y_train.to_csv(os.path.expanduser("data/y_train.csv"), index=False)
y_test.to_csv(os.path.expanduser("data/y_test.csv"), index=False)


## LightGBM

LightGBM construit chaque arbre feuille par feuille. XGBoost le construit niveau par niveau. Sans élagage, les deux partitions coïncident. `max_depth` limite la profondeur.
GOSS conserve en priorité les observations à fort gradient, puis complète l'échantillon.
`min_child_samples` limite l'effectif d'une feuille lorsque `num_leaves` est élevé.
La validation croisée porte sur `(X, y)`. L'ajustement final porte sur le partage entraînement / test.


In [ ]:
from lightgbm.sklearn import LGBMClassifier
from sklearn.model_selection import RepeatedKFold
from sklearn.metrics import f1_score


In [ ]:
rep_kfold = RepeatedKFold(n_splits=4, n_repeats=3)

hyp_params = {
    "num_leaves": 60,
    "min_child_samples": 10,
    "max_depth": 12,
    "n_estimators": 100,
    "learning_rate": 0.1,
}


In [ ]:
scores_train = []
scores_test = []
n_iter = 1
for train_I, test_I in rep_kfold.split(X):
    print("Itération {} du k-Fold".format(n_iter))
    X_fold_train = X.iloc[train_I, :]
    y_fold_train = y.iloc[train_I]
    X_fold_test = X.iloc[test_I, :]
    y_fold_test = y.iloc[test_I]

    model = LGBMClassifier(**hyp_params, objective="binary", verbose=-1)
    model.fit(X_fold_train, y_fold_train)

    scores_train.append(f1_score(y_fold_train, model.predict(X_fold_train)))
    scores_test.append(f1_score(y_fold_test, model.predict(X_fold_test)))
    n_iter += 1


In [ ]:
import matplotlib.ticker as mtick

print("Score Train médian : {:2.1f}%".format(np.mean(scores_train) * 100))
print("Score Test médian : {:2.1f}%".format(np.mean(scores_test) * 100))

scores = pd.DataFrame.from_dict({
    "Train Set": scores_train,
    "Test Set": scores_test,
})

plt.figure(figsize=(14, 4))
sns.boxplot(data=scores, orient="h")
plt.title("Scores des modèles du k-Fold", fontsize=17)
plt.gca().xaxis.set_major_formatter(mtick.PercentFormatter(1, 1))
plt.show()


In [ ]:
model = LGBMClassifier(**hyp_params, objective="binary", verbose=-1)
model.fit(X_train, y_train)


In [ ]:
print("Score Train : {:2.1f}%".format(f1_score(y_train, model.predict(X_train)) * 100))
print("Score Test : {:2.1f}%".format(f1_score(y_test, model.predict(X_test)) * 100))


## Précision et rappel

Le seuil de décision vaut 0,5 sur la probabilité d'achat.
La spécificité est TN / (FP + TN). Le rappel est TP / (TP + FN). La précision est TP / (TP + FP).
Un faux négatif a l'impact économique le plus fort : la remise touche un client qui aurait acheté, et un achat manqué coûte plus qu'une remise inutilisée.
Le F1 est la moyenne harmonique de la précision et du rappel. Un modèle qui prédit toujours un achat obtient un rappel de 1 et une précision égale à la prévalence.


In [ ]:
from sklearn.metrics import recall_score, precision_score

print("Precision Test : {:2.1f}%".format(precision_score(y_test, model.predict(X_test)) * 100))
print("Recall Test : {:2.1f}%".format(recall_score(y_test, model.predict(X_test)) * 100))
